# Social Media Sentiment and User Engagement Analysis

Link to dataset: [https://www.kaggle.com/datasets/kashishparmar02/social-media-sentiments-analysis-dataset/data].

In this project, I analyze the sentiment, topic and user engagement behavior from comments on social media. The dataset is from Kaggle, capturing user-generated social media content. The key variables include encompassing text, timestamps, hashtags, countries, likes, and retweets. The goal is to explore the emotional tone of the text, users' topic preferences, and their interaction behavior. This project should shed light on market research. 

Here is the outline of the project.
1. Download the dataset
2. Exploratory data analysis
3. Preprocessing
4. Sentiment analysis
5. Topic Modeling
6. User engagement analysis
7. Conclusions

## Step 1. Download the Dataset

Steps:

- Install required libraries and set up plot options
- Download data from Kaggle
- View dataset files
- Load the head of the data set with Pandas
- Display the basic information of the data set

In [ ]:
# import libraries and set up plot options
import os 
#os.chdir('C:/Users/wangd/Dropbox/2025.08 onwards')
os.chdir('C:/Users/wangd/Dropbox/2025.08 onwards/social-media-sentiments-analysis-dataset')
import pyarrow.parquet as pq
import opendatasets as od
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re
import seaborn as sns
from collections import Counter
import geopandas as gpd
import nltk
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from itertools import combinations

%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (10, 6)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

In [ ]:
# download data from Kaggle
dataset_url = 'https://www.kaggle.com/datasets/kashishparmar02/social-media-sentiments-analysis-dataset/data'
#%%time
od.download(dataset_url)

In [ ]:
# view the download folder
data_dir = '.\social-media-sentiments-analysis-dataset'
!ls -lh {data_dir}

In [ ]:
# load the dataset and display the first 10 rows
df = pd.read_csv(data_dir+"/sentimentdataset.csv")
df.head(10)

In [ ]:
# basic information about the numerical columns
df.describe()

In [ ]:
# basic information about the string columns
df.describe(include = ['object'])

In [ ]:
# check the number of missing values
print(df.isna().sum())
print(df.eq('').sum())

In [ ]:
# check the number of duplicated rows
df.duplicated().sum()

Some observations about the dataset:
- There are 732 columns of data, including text-related variables such as text, sentiment, user, platform, user id,country, and hashtag; time-related variables such as timestamp, year, month, day, and hour; engagement-related variables such as retweets and likes.
- There are no missing values or duplicated rows.
- There are more than 700 texts by 685 unique users on 4 platforms.Users are from 115 countries.The years are from 2010-2023.
- Most texts were posted from the USA. Most texts were from Instagram.Most texts have hashtags.
- There are 279 unique types of sentiment. Most are positive.
- On average, a text receives 42 likes and 23 retweets.

Overall, this dataset provides rich characteristics for further analysis.

## Step 2. Exploratory Data Analysis

This section answers the following questions.
1. Geography: What is the distribution of the number of texts, sentiment, and hashtags over countries?
2. Time: Are there annual/seasonal/hour-of-day posting patterns in the number of texts and hashtags?
3. Length: Distribution of text length in characters and words?
4. Text/Hashtag content: What words appear most often? What hashtags appear most often? 
5. Sentiment: What hashtags and sentiment tend to appear together? 
6. User: Distribution of the number of texts per user?
7. Engagement: Do certain times of day/countries yield more engagement? Correlation between sentiment/hashtag and engagement?


Before diving into the questions, let's process the hashtag column to extract the words, and sentiment column to group the emotions.
- Hashtags: Each text contains 2 hashtags. Some hashtags include multiple words like "PersonalGrowth", and I have 2 versions of separating and keeping them. I also remove the whitespace and "#".
- Sentiment: I first remove the white space and change them to lower case. Using a lexicon-based approach, I map the sentiment column to 3 sentiment groups (positive, negative, neutral), and then to Plutchik's 8 emotion categories (anger, anticipation, disgust, fear, joy, sadness, surprise, trust). Those that cannot be mapped is grouped to "other" category. The data comes from National Research Council Canada (NRC) (access [here](http://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm)). It contains ~14,000 English words, each tagged as associated (1) or not (0) with the sentiment and emotion categories.

In [ ]:
df.drop(columns = {'Unnamed: 0.1', 'Unnamed: 0'}, inplace = True)

def camel_split(token: str):
    # "PersonalGrowth" -> ["Personal","Growth"]; keeps acronyms like "FDA"
    parts = re.sub(r'([a-z])([A-Z])', r'\1 \2', token).split()
    return parts if len(parts) > 1 else [token]

def clean_and_segment_hashtags(s):
    if pd.isna(s) or not str(s).strip():
        return [], []
    # split on whitespace, strip '#'
    raw = [t.lstrip('#').strip() for t in str(s).split() if t.strip()]

    words = []
    for tag in raw:
        parts = camel_split(tag)
        words.extend(p.lower() for p in parts if p)
    return raw, words

# Example: one column "hashtags" like "#PersonalGrowth #FamilyLaughter"
df[["hashtags_original", "hashtags_words"]] = df["Hashtags"].apply(
    lambda x: pd.Series(clean_and_segment_hashtags(x))
)

all_tags = sum(df["hashtags_original"], [])  # flatten list of lists
tag_counts = Counter(all_tags).most_common(10)
tag_counts


In [ ]:
# Load NRC Emotion Lexicon (after downloading & unzipping)
# Format is usually: word, emotion, association (0/1)
nrc = pd.read_csv("NRC-Emotion-Lexicon-Wordlevel-v0.92.txt",
                  sep="\t", names=["word","emotion","association"])
# keep sentiment and positive association
nrc_group3 = nrc[((nrc['emotion'] == 'negative') | (nrc['emotion'] == 'positive')) & (nrc['association'] == 1)]

# Build dictionary: word -> list of emotions
lexicon_map_group3 = (
    nrc_group3.groupby("word")["emotion"]
    .first()   
    .to_dict()
)

def map_to_sentiment(sentiment):
    if sentiment == 'positive':
        return 'positive'
    elif sentiment == 'negative':
        return 'negative'
    elif sentiment == 'neutral':
        return 'neutral'
    word = sentiment.strip().lower()
    return lexicon_map_group3.get(word, "other")   # fallback if not in lexicon

# Remove spaces in Sentiment and apply mapping
df['Sentiment'] = df["Sentiment"].str.strip().str.lower()
df["mapped_sentiment"] = df["Sentiment"].apply(map_to_sentiment)


In [ ]:
# Keep emotions and positive association
nrc_group8 = nrc[((nrc['emotion'] != 'negative') & (nrc['emotion'] != 'positive')) & (nrc['association'] == 1)]

# Build dictionary: word -> list of emotions
lexicon_map_group8 = (
    nrc_group8.groupby("word")["emotion"]
    .first()   
    .to_dict()
)

def map_to_emotion(sentiment):
    if sentiment == 'positive':
        return 'joy'
    elif sentiment == 'negative':
        return 'sadness'
    elif sentiment == 'neutral':
        return 'other'
    word = sentiment.strip().lower()
    return lexicon_map_group8.get(word, "other")   # fallback if not in lexicon

# Apply mapping

df["mapped_emotion"] = df["Sentiment"].apply(map_to_emotion)


In [ ]:
# The distribution of mapped sentiment and emotions
print(df['mapped_sentiment'].value_counts())
print(df['mapped_emotion'].value_counts())

Some observations:
- Many hashtags are emotions too, which is useful for analysis. The top hashtags are serenity, graditude, and excitement.
- The sentiment is mainly positive (about half), followed by other and negative. Neutral sentiment is rare.
- About 1/3 of emotions are other. Still, the information is useful, as more than 150 data points have joy or anticipation respectively, followed by anger, sadness, and fear.
Now we are ready for EDA.
### 1. Geography: What is the distribution of the number of texts, sentiment, and hashtags over countries?

In [ ]:
df['Country'] = df["Country"].str.strip()

country_table = pd.read_excel(data_dir+"/country.xlsx")
country_dict = dict(zip(country_table['Country'],country_table['country_ISO3']))

df['country_ISO3'] = df['Country'].map(country_dict)


df.loc[df["Country"] == "USA", "country_ISO3"] = "USA"
df.loc[df["Country"] == "UK", "country_ISO3"] = "GBR"
df.loc[df["Country"] == "Scotland", "country_ISO3"] = "GBR"

In [ ]:
# 1) Aggregate counts by country
posts_by_cty = df.groupby("country_ISO3").size().reset_index(name="count")

# 2) Load a world shapefile 
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# 3) Harmonize names
merged = world.merge(posts_by_cty, how="left", left_on="ADM0_A3", right_on="country_ISO3")

# 4) Plot choropleth
fig, ax = plt.subplots(figsize=(12,6))
merged.plot(column="count",cmap="GnBu",  ax=ax, legend=True, missing_kwds={"color":"lightgrey"})
ax.set_title("Number of Posts by Country")
ax.axis("off")

top_countries = posts_by_cty.sort_values("count", ascending=False).head(10)

for _, row in merged.iterrows():
    if row["country_ISO3"] in top_countries["country_ISO3"].values:
        plt.text(
            row["geometry"].centroid.x,
            row["geometry"].centroid.y,
            row["country_ISO3"], fontsize=6, ha="center", color = 'coral'
        )

plt.show()

- The top posting countries are USA, Canada, UK, Australia, and India. USA has more than 175 posts.

In [ ]:
top_cty_list = (df.groupby("country_ISO3").size()
                  .sort_values(ascending=False)
                  .head(5).index.tolist())

# Filter to top countries
df_top = df[df["country_ISO3"].isin(top_cty_list)].copy()

# Build counts per country × sentiment
pivot = (df_top
         .pivot_table(index="country_ISO3", columns="mapped_sentiment", values = "Text",
                      aggfunc="count", fill_value=0))

# Ensure consistent column order
for col in ["positive","neutral","negative"]:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[["positive","neutral","negative"]]

# Normalize to proportions (optional)
prop = pivot.div(pivot.sum(axis=1), axis=0)

# Plot stacked bars (proportions)
countries = prop.index.tolist()
x = np.arange(len(countries))

plt.figure(figsize=(8,6))
bottom = np.zeros(len(countries))
for col in ["positive","neutral","negative"]:
    plt.bar(countries, prop[col].values, bottom=bottom, label=col, width = 0.4)
    bottom += prop[col].values

plt.xticks(rotation=45, ha="right")
plt.ylabel("Proportion")
plt.title(f"Sentiment Distribution (Top 5 Countries)")
plt.legend(bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()


- All countries have more than 50% positive sentiment. The lowest is India, with less than 60%. Neutral sentiment is very rare across countries. 

In [ ]:
topK_tags = 5

top_cty_list = (df.groupby("country_ISO3").size()
                  .sort_values(ascending=False)
                  .head(5).index.tolist())

rows = []
for c in top_cty_list:
    # explode hashtag list column
    sub = df[df["country_ISO3"]==c].explode("hashtags_original")
    # clean empties
    sub = sub[sub["hashtags_original"].notna() & (sub["hashtags_original"].str.strip()!="")]
    top_tags = (sub["hashtags_original"].str.strip()
                  .str.lstrip("#").str.lower()
                  .value_counts().head(topK_tags))
    for tag, cnt in top_tags.items():
        rows.append({"country_ISO3": c, "hashtag": tag, "count": cnt})

top_tags_tbl = pd.DataFrame(rows)
# This table is great to display or export to CSV
# top_tags_tbl.to_csv("top_hashtags_by_country.csv", index=False)
top_tags_tbl

- The table displays the top 5 hashtags for the top 5 posting countries. Every country has a different set of trending hashtags. The only hashtags that appeared twice are confusion, contentment, and curiosity. Most are positive, corresponding to the results above. It suggests hashtags are useful information to predict sentiment, but we need to be careful that the machine may read sentiment directly from hashtags.

### 2. Time: Are there annual/seasonal/hour-of-day posting patterns in the number of texts and sentiment?

In [ ]:
# --- Parse timestamp & derive time columns (robust if not yet present) ---
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df = df.dropna(subset=["Timestamp"]).copy()

df["Year"] = df["Timestamp"].dt.year
df["Month"] = df["Timestamp"].dt.month
df["Day"] = df["Timestamp"].dt.date
df["Hour"] = df["Timestamp"].dt.hour
df["Week"] = df["Timestamp"].dt.to_period("W").apply(lambda r: r.start_time)  # week start
df["Weekday"] = df["Timestamp"].dt.day_name()  # Monday..Sunday

# Optional: order weekdays nicely
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df["Weekday"] = pd.Categorical(df["Weekday"], categories=weekday_order, ordered=True)


In [ ]:
year_sent = (df.groupby(["Year","mapped_sentiment"])
             .size()
             .reset_index(name="count"))
year_sent_pivot = year_sent.pivot(index="Year", columns="mapped_sentiment", values="count").fillna(0)
year_sent_long = year_sent_pivot.reset_index().melt(id_vars="Year", var_name="sentiment", value_name="count")

fig = px.area(year_sent_long, x="Year", y="count", color="sentiment",
              title="Annual Sentiment Volume (Stacked)")
fig.update_layout(yaxis_title="Posts", xaxis_title="Year")
fig.show()


- The number of posts each year is low up to 2015, and increases from 2014 to 2016. The rate of increase is higher from 2016 to 2019. Then the number declines from 2019 to 2022. It increases sharply after 2022.
- The number of positive posts have an upward trend frim 2014 to 2019, as well as after 2022. It's stable from 2019 to 2022. The number of negative posts increases from 2015 to 2021, and drops from 2021 to 2022, and increases again. The number of neutral posts is low, and starts increasing from 2022.

In [ ]:
monthly_sent = df.groupby(["Month","mapped_sentiment"]).size().reset_index(name="count")
fig = px.bar(monthly_sent, x="Month", y="count", color="mapped_sentiment",
             title="Seasonality: Sentiment by Month", barmode="relative")
fig.update_layout(yaxis_title="Posts", xaxis=dict(tickmode="linear"))
fig.show()


- The number of posts per month is the highest in February, followed by January, August, and September. The lowest month is December.
- Positive posts have a higher proportion in January, July, and August, accounting for more than 50%. Negative posts have a higher proportion in September, February, March, and May.

In [ ]:
hour_weekday = (df.groupby(["Weekday","Hour"])
                  .size()
                  .reset_index(name="count"))
fig = px.density_heatmap(hour_weekday, x="Hour", y="Weekday", z="count",
                         title="Posts by Weekday and Hour", nbinsx=24, histfunc="sum",
                         color_continuous_scale="Blues")
fig.update_layout(xaxis=dict(dtick=1))
fig.show()


- The most frequent posting period if afternoon (2 pm to 7 pm). Within this period, the highest number of posts appears at 2 pm on Friday, Saturday, and Tuesday, as well as 6 pm on Friday.

In [ ]:
agg = (
    df.groupby(["Weekday","Hour"])["mapped_sentiment"]
      .agg(total="count", positive=lambda s: (s=="positive").sum())
      .reset_index()
)
agg["prop_positive"] = agg["positive"] / agg["total"]

# Ensure full 7×24 grid exists (fill missing with zero)
grid = pd.MultiIndex.from_product([weekday_order, range(24)], names=["Weekday","Hour"]).to_frame(index=False)
agg = grid.merge(agg, on=["Weekday","Hour"], how="left").fillna({"total":0, "positive":0, "prop_positive":0})

# --- 3) Pivot to matrix for imshow ---
mat = (agg.pivot(index="Weekday", columns="Hour", values="prop_positive")
         .reindex(weekday_order))

# --- 4) Plot ---
fig = px.imshow(
    mat,
    aspect="auto",
    color_continuous_scale="Reds",   # change palette here if you like
    origin="upper",
    title="Proportion of Positive Sentiment by Weekday × Hour"
);
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Weekday",
    coloraxis_colorbar=dict(title="Proportion", tickformat=".0%")
);
fig.update_traces(hovertemplate="Weekday=%{y}<br>Hour=%{x}<br>Positive=%{z:.2%}<extra></extra>");
fig.show()


- A high proportion of positive sentiment appear in late night and early morning (10 pm to 7 am). Some hours during the day also see high positive sentiment such as Monday and Wednesday around noon.

### 3. Length: Distribution of text length in characters and words?

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['Text'].str.len(), bins = 10);

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(df['Text'].apply(lambda x: len(str(x).split())), bins = 10);

- Most texts are less than 60 characters, followed by 60-70 and 90-100. Very few texts exceed 140 characters.
- Most texts have around 7-20 words.

### 4. Text/Hashtag content: What words appear most often? What hashtags appear most often?
To do this, we first remove the stopwords in the text.

In [ ]:
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", " ", text)     # remove URLs
    text = re.sub(r"[^a-z\s]", " ", text)    # keep only letters
    return text

df["text_clean"] = df["Text"].apply(clean_text)

In [ ]:
# join all text together
all_words = " ".join(df["text_clean"].dropna())

wordcloud = WordCloud(
    width=800, height=400,
    background_color="white",
    stopwords=stop_words,
    max_words=200,
    colormap="viridis"   # try 'plasma', 'coolwarm', 'Set2'…
).generate(all_words)

# plot
plt.figure(figsize=(10,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Common Words in Posts")
plt.show()


In [ ]:
# Generate word cloud
wordcloud = WordCloud(width=800, height=400, background_color="white").generate(" ".join(all_tags))

# Display
plt.figure(figsize=(10,6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Hashtag Word Cloud")
plt.show()


- Words that appear most often include life, new, joy, day, feeling, friend, moment, heart, etc.
- Positive hashtags such as excitement, gratitude, serenity, contentment, and nostalgia appear the most often. Commonly seen negative hashtags include loneliness, despair and grief.

### 5. What hashtags and sentiment tend to appear together?

In [ ]:
df_tags = df.explode("hashtags_original").dropna(subset=["hashtags_original"]).copy()

# Clean hashtag text
df_tags["hashtags_original"] = (
    df_tags["hashtags_original"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.lstrip("#"))

# Count per (hashtag, sentiment)
counts = (
    df_tags.groupby(["hashtags_original","mapped_sentiment"])
    .size()
    .reset_index(name="count")
)

# Also compute totals per hashtag for ranking
totals = counts.groupby("hashtags_original")["count"].sum().reset_index(name="total")
counts = counts.merge(totals, on="hashtags_original")

topN = 30
top_tags = totals.sort_values("total", ascending=False).head(topN)["hashtags_original"]

counts_top = counts[counts["hashtags_original"].isin(top_tags)]


fig = px.bar(
    counts_top,
    x="hashtags_original",
    y="count",
    color="mapped_sentiment",
    title=f"Top {topN} Hashtags by Sentiment Distribution",
    barmode="stack"
)

fig.update_layout(
    xaxis_title="Hashtag",
    yaxis_title="Post Count",
    xaxis_tickangle=-45
)
fig.show()


- Positive sentiment is associated with hashtags such as serenity, excitement, and gratitude. Negative sentiment appears together with hashtags like despair, grief, and loneliness. Other sentiment appears together with hashtgas like contentent and nostalgia.

### 6. User: Distribution of the number of texts per user? 

In [ ]:
user_counts = df["User"].value_counts().reset_index()
user_counts.columns = ["user", "count"]

fig = px.histogram(
    user_counts,
    x="count",
    nbins=50,
    title="Distribution of Number of Posts per User",
);
fig.update_layout(xaxis_title="Posts per User", yaxis_title="Number of Users");
fig.show()

- Most users have one post; 35 users have 2 posts, and 6 users have 3 posts. There are no top users that have many posts.

### 7. Engagement: Do certain times of day/countries yield more engagement? Correlation between sentiment/hashtag and engagement?

In [ ]:
df.groupby("Hour")[["Likes","Retweets"]].mean().sort_values("Likes", ascending=False)

In [ ]:
df.groupby("Country")[["Likes","Retweets"]].mean().sort_values("Likes", ascending=False).head(20)

In [ ]:
df.groupby("mapped_sentiment")[["Likes","Retweets"]].agg(["mean","median"])


In [ ]:
df_tags.groupby("hashtags_original")[["Likes","Retweets"]].mean().sort_values("Likes", ascending=False).head(20)

- On average, posts at night or early evening (11 pm, 5 am, 10 pm, 8 pm) get most likes and retweets.
- Posts from South Africa, Belgium, Thailand, Sweden, Jamaica and Jordan get most likes and retweets.
- On average, a positive post gets 45 likes and 22 retweets; a negative post gets 34 likes and 17 retweets.
- Quite a few hashtags get 80 likes and 40 retweets, which is the highest number in the dataset. Some of these hashtags are soccer defeats, sunset beauty, mesmerizing, and rare book discovery.

## Step 3. Preprocessing
In this step, we preprocess the data so it's ready for sentiment analysis. We have removed stopwords and created a new column named `text_clean`, as well as generated time-related features like `Year`, `Month`, `Day`, `Hour`, `Week`, and `Weekday`, and split the hashtags. We will conduct text analysis including tokenization, lowercasing, and lemmatization. Then we will generate n-grams (bi- or tri-grams in this case), and compute the TF-IGF score.

### Logistic Regression with TF-IDF score
The inputs are TF‑IDF with unigrams + bigrams, so that the model captures short phrases (“not good”, “very happy”) that single tokens miss. We generate column `clean_text`that removes orls, mentions, non-alphabetic characters, stopwords, and lemmatize. Then use vectorizer to build word-level TF-IDF (unigrams + bigrams) and character-level TF-IDF (3–5 character n-grams).

In [ ]:
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def clean_text(text, lemmatize=True, remove_stopwords=True):
    if not isinstance(text, str):
        return ""
    # lowercase
    text = text.lower()
    # remove urls
    text = re.sub(r"http\S+|www\S+", " ", text)
    # remove mentions and hashtags (optional: keep hashtags if useful)
    text = re.sub(r"@\w+", " ", text)
    # keep hashtags as tokens (strip only the '#')
    text = text.replace("#", " ")
    # remove non-alphabetic chars
    text = re.sub(r"[^a-z\s]", " ", text)
    # tokenize
    tokens = nltk.word_tokenize(text)
    # remove stopwords if needed
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    # lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

# Apply to your dataframe
df['text_hash'] = df["Text"] + ' ' + df['Hashtags']
df["text_clean"] = (df['text_hash']).apply(clean_text)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# Word-level TF-IDF (unigrams + bigrams)
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1,2),        # unigrams + bigrams
    max_features=50000,       # cap vocab size
    min_df=3                  # ignore rare words
)

# Character-level TF-IDF (3–5 char n-grams)
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    max_features=30000        # cap to avoid explosion
)

# Combine word and char features
vectorizer = FeatureUnion([
    ("word", word_vectorizer),
    ("char", char_vectorizer)
])

# Fit + transform
X = vectorizer.fit_transform(df["text_clean"])  # from your preprocessing step
y = df["mapped_sentiment"]   # target


### LSTM
The inputs of LSTM are token sequences instead of n-gram features. We keep raw surface forms without heavy preprocessing, apply tokenization and padding.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# For LSTM: keep more raw surface forms, no heavy preprocessing
def prepare_text_for_lstm(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    return text

df["text_lstm"] = df["text_hash"].apply(prepare_text_for_lstm)

# Tokenize & pad
max_words = 30000  # vocab size cap
max_len = 100      # truncate/pad to 100 tokens per doc

tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(df["text_lstm"])

sequences = tokenizer.texts_to_sequences(df["text_lstm"])
X_lstm = pad_sequences(sequences, maxlen=max_len, padding="post", truncating="post")

## Step 4. Sentiment Analysis
In this section, we conduct and evaluate sentiment analysis of 2 models: logistic regression with TF-IDF, and LSTM model. We compare their performance in 4 dimensions: accuracy, precision, recall, and f1 score.
- Accuracy measure the model's quality of a prediction (either positive or negative). It's defined as `(true positives + true negatives) / (true positives + true negatives + false positives + false negatives)`.
- Precision measures the model's quality of a positive prediction. It's defined as `true positives / (true positives + false positives)`.
- Recall measures the model's ability to find all true positives. It's defined as `true positives / (true  positives + false negatives)`. 
- F1 score is a trade-off between precision and recall, defined by `2 * (precision * recall) / (precision + recall)`. If either precision or recall is 0, then F1 score is 0, so it penalizes the extreme negative values of either component.

### Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))


The logistic model performs quite well for both positive and negative sentiment categories, followed by the other category. It performs poorly for the neutral category, which may be explained by a lack of data (supported only by 4 cases).
- The accuracy is 0.78, meaning the model correctly predicts 78% of the test set.
- The model achieves 0.90 precision for the nagative sentiment, followed by positive (0.78), and other (0.71). The neutral category has low precision (0.5).
- The model is able to find 84% true positives of the positive sentiment, and 82% true positives of the negative sentiment, followed by other (69%). Again, the neutral category has low recall (0.25).
- The F1 score is high for both negative and positive sentiment categories (86% and 81%), followed by other (70%). Neutral category has 0.33 f1-score.
- The confusion matrix compares the predicted versus actual outputs of the model. 


### LSTM

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight


text_col = "text_lstm"        
label_col = "mapped_sentiment" #

le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))
X_train_lstm, X_temp, y_train_lstm, y_temp = train_test_split(
    df[text_col], y, test_size=0.2, random_state=42, stratify=y
)
X_val_lstm, X_test_lstm, y_val_lstm, y_test_lstm = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


classes = np.unique(y_train_lstm)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train_lstm)
class_weights = {int(c): float(w) for c, w in zip(classes, class_weights)}


import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 30000
tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(X_train_lstm.tolist())

def to_seq(s):
    s = s if isinstance(s, str) else ""
    s = s.lower()
    s = re.sub(r"http\S+|www\S+", " ", s)
    s = re.sub(r"@\w+", " ", s)
    return s

def seqs(texts):
    seq = tokenizer.texts_to_sequences([to_seq(t) for t in texts])
    return seq

len_train = [len(x) for x in tokenizer.texts_to_sequences([to_seq(t) for t in X_train_lstm])]
max_len = int(np.percentile(len_train, 90))  # e.g., 90th percentile

Xtr = pad_sequences(seqs(X_train_lstm), maxlen=max_len, padding="post", truncating="post")
Xva = pad_sequences(seqs(X_val_lstm),   maxlen=max_len, padding="post", truncating="post")
Xte = pad_sequences(seqs(X_test_lstm),  maxlen=max_len, padding="post", truncating="post")

num_classes = len(le.classes_)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
emb_dim = 100
emb_matrix = np.random.normal(0, 0.6, size=(max_words, emb_dim)).astype("float32")
user_pretrained = False
def build_lstm(max_words, max_len, num_classes, emb_dim=100, emb_matrix=None, train_emb=True, dropout=0.3):
    inp = layers.Input(shape=(max_len,), dtype="int32")
    if emb_matrix is not None and use_pretrained:
        emb = layers.Embedding(max_words, emb_dim, weights=[emb_matrix],
                               input_length=max_len, trainable=train_emb)(inp)
    else:
        emb = layers.Embedding(max_words, emb_dim, input_length=max_len)(inp)

    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(emb)
    x = layers.SpatialDropout1D(0.2)(x)
    gmp = layers.GlobalMaxPooling1D()(x)
    gap = layers.GlobalAveragePooling1D()(x)
    x = layers.Concatenate()([gmp, gap])
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout)(x)

    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_lstm(max_words, max_len, num_classes, emb_dim=emb_dim,
                   emb_matrix=None,
                   train_emb=True, dropout=0.4)
model.summary()


In [ ]:
cb = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1)
]

history = model.fit(
    Xtr, y_train_lstm,
    validation_data=(Xva, y_val_lstm),
    epochs=10,
    batch_size=128,
    class_weight=class_weights,   # helps with “neutral”; remove if not needed
    callbacks=cb,
    verbose=1
)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(Xte, batch_size=256).argmax(axis=1)
print(classification_report(y_test_lstm, y_pred, target_names=le.classes_))
print(confusion_matrix(y_test_lstm, y_pred))


- Accuracy: The LSTM achieves 69% accuracy, lower than the logistic regression baseline (78%).
- Precision: LSTM shows high precision for negative (1.00) and positive (0.88) categories, moderate for other (0.56), and low for neutral (0.25).
Compared with logistic regression, LSTM is more precise on positive/negative, but less precise on neutral/other.
- Recall: LSTM recalls neutral and other categories relatively well, but under-recalls positive and negative.
This contrasts with precision: the model over-predicts neutral/other (false positives) and under-predicts positive/negative (false negatives).
Compared with logistic regression, LSTM achieves higher recall on neutral/other, but lower recall on positive/negative.
- F1 score:  LSTM has balanced F1 scores (around 0.7) for positive, negative, and other, but lower for neutral (around 0.4).
Relative to logistic regression: Better on neutral, Similar on other, Worse on positive/negative.

#### Summary
Overall, logistic regression with TF–IDF remains the stronger model in this dataset: it leverages sparse n-grams effectively, making it robust with limited data (~700 samples).
LSTM underperforms in accuracy, likely due to small sample size and higher parameterization needs.
Still, the two models complement each other: logistic regression is stronger on positive/negative sentiment, while LSTM finds more neutral/other cases.

In [ ]:
df.to_parquet(data_dir + '/df_pq.parquet')

In [ ]:
# read the parquet file
df = pq.read_table('df_pq.parquet')
df = df.to_pandas()


## Step 5. Topic Modeling

Topic modeling is an unsupervised learning approach that discover groups of words that often appear together (topics). A post is a mixture of topics. Topic modeling is useful to summarize a large collection of texts, explore themes over time or across groups, and produce inputs for further analysis. 
In this section, I use two methods to explore the common topics in this dataset.
1.  Latent Dirichlet Allocation (LDA), which is classic probabilistic approach. In this method, each topic is a distribution over words, and each document is a distribution over topics.
2.  Non-negative Matrix Factorization (NMF), which is a linear algebra approach. It decomposes the document-term matrix into interpretable factors.

The steps are as follows.
- Preprocessing (already done).
- Build document-term matrix. The rows are posts, the columns are words or n-grams, and the entries are TF-IDF scores or counts.
- Fit to the model to produce the word distribution over topics.
- Assign topics back to the document, so each post has a distribution over topics.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.preprocessing import normalize
# A) Count (for LDA)
count_vect = CountVectorizer(
    ngram_range=(1,2),       # unigrams + bigrams help short texts
    max_df=0.7,              # drop very common terms
    min_df=5                 # drop very rare terms
)
X_count = count_vect.fit_transform(df["text_hash"])
vocab_count = np.array(count_vect.get_feature_names_out())

# B) TF-IDF (for NMF)
tfidf_vect = TfidfVectorizer(
    ngram_range=(1,2),
    max_df=0.7,
    min_df=5,
    norm="l2"
)
X_tfidf = tfidf_vect.fit_transform(df["text_hash"])
vocab_tfidf = np.array(tfidf_vect.get_feature_names_out())


In [ ]:
def show_topics(topic_word, vocab, topn=12, title="Topics"):
    print(f"\n{title}\n" + "-"*len(title))
    for i, comp in enumerate(topic_word):
        top_idx = comp.argsort()[::-1][:topn]
        terms = [vocab[j] for j in top_idx]
        print(f"Topic {i:02d}: " + ", ".join(terms))

Then we fit both LDA and NMF models. We compare across a range of `k` (the number of topics). The metrics to evaluate LDA is model perplexity (lower is better), and the metric to evaluate NMF is reconstruction error. However, ultimately we should choose `k` that is the most interpretable.

In [ ]:
def fit_lda_k(X_count, vocab, k_list=(6,8,10,12), topn=10,
              max_iter=20, learning_method="batch", random_state=42,
              evaluate_perplexity=True, X_holdout=None):
    """
    Parameters
    ----------
    X_count : csr_matrix
        Document-term **count** matrix (use CountVectorizer).
    vocab : np.ndarray or list
        Feature names from the vectorizer (len = n_features).
    k_list : iterable
        Topic numbers to try.
    topn : int
        # of top words to display per topic.
    max_iter : int
        LDA training iterations.
    learning_method : {"batch","online"}
        scikit-learn LDA solver.
    evaluate_perplexity : bool
        If True, print train perplexity; if X_holdout provided, also held-out.
    X_holdout : csr_matrix or None
        Optional held-out set to compute **held-out perplexity** (better for model selection).

    Returns
    -------
    results : dict
        k -> dict with {"model","doc_topic","topic_word","train_perplexity","train_loglik",
                        "heldout_perplexity"(opt)}
    """
    results = {}
    for k in k_list:
        lda = LatentDirichletAllocation(
            n_components=k,
            learning_method=learning_method,
            max_iter=max_iter,
            random_state=random_state,
            evaluate_every=0  # we’ll compute at the end
        )
        doc_topic = lda.fit_transform(X_count)       # document-topic distributions
        topic_word = lda.components_                 # topic-word weights

        # Metrics
        train_loglik = lda.score(X_count)            # higher is better
        train_perp   = lda.perplexity(X_count) if evaluate_perplexity else None

        heldout_perp = None
        if evaluate_perplexity and X_holdout is not None:
            heldout_perp = lda.perplexity(X_holdout)

        print(f"\n=== LDA k={k} ===")
        print(f"log-likelihood (train): {train_loglik:.2f}")
        if train_perp is not None:
            print(f"perplexity (train):  {train_perp:.2f}")
        if heldout_perp is not None:
            print(f"perplexity (held-out): {heldout_perp:.2f}")

        show_topics(topic_word, vocab, topn=topn, title=f"LDA Topics (k={k})")

        results[k] = {
            "model": lda,
            "doc_topic": doc_topic,
            "topic_word": topic_word,
            "train_loglik": train_loglik,
            "train_perplexity": train_perp,
            "heldout_perplexity": heldout_perp
        }
    return results


In [ ]:
results_lda = fit_lda_k(
    X_count, vocab_count,
    k_list=(6,8,10,12), topn=12,
    max_iter=20, learning_method="batch",
    evaluate_perplexity=True,
    # X_holdout=X_holdout_c  # uncomment if you created a held-out matrix
)

# Access best model (e.g., by lowest perplexity or best interpretability)
best_k = min(results_lda, key=lambda k: results_lda[k]["train_perplexity"])
best_lda = results_lda[best_k]["model"]
doc_topic = results_lda[best_k]["doc_topic"]


- The `k` that has the lowest model perplexity is 8. It is also interpretable, including topics like life, project, nature, echoes, nostalgia, loneliness, and hope. It seems the results when `k` is 10 are more interpretable: there are topics like new, friend, nature, world, art, joy, community, symphony, bitterness, dreams.

In [ ]:
def fit_nmf_k(X, vocab, k_list=(6,8,10,12), topn=10):
    results = {}
    for k in k_list:
        nmf_k = NMF(n_components=k, init="nndsvda", random_state=42, max_iter=400)
        W = nmf_k.fit_transform(X)
        H = nmf_k.components_
       # results[k] = (W, H, nmf_k.reconstruction_err_)
        print(f"\n=== NMF k={k} (recon error={nmf_k.reconstruction_err_:.4f}) ===")
        show_topics(H, vocab, topn=topn, title=f"NMF k={k}")
        results[k] = {
            "model": nmf_k,
            "doc_topic": W,
            "topic_word": H,
            "reconstruction error": nmf_k.reconstruction_err_}
    return results

# Example
results_nmf = fit_nmf_k(X_tfidf, vocab_tfidf, k_list=(6, 8,10,12))
best_k_nmf = min(results_nmf, key=lambda k: results_nmf[k]["reconstruction error"])
best_nmf = results_nmf[best_k_nmf]["model"]
doc_topic_nmf = results_nmf[best_k_nmf]["doc_topic"]

- The k that has the lowest model reconstruction error is 12. It is also interpretable, including topics like life, school, serenity, community, friends, embark, local, projects, anxiety, curiosity, nature, and excitement. When `k` = 10, the topics are life, joy/school, serenity, gratitude, friends, nature, local, project, anxiety, symphony.

- Compared with LDA, both include topics like friend, nature, joy, community and symphony. This means these topics are truly important in the data. The other topics differ, which means either the patterns are weak or the 2 models have different angles.

- We experimented with different topic numbers. At k=8, LDA produced broader topics (life, nature, community). At k=10–12, LDA and NMF split these into finer categories (nostalgia, curiosity, projects). We use k=10 for downstream analysis as it balances coherence and detail.

In [ ]:
k = 10  # try 8, 10, 12 and compare interpretability

# A) LDA (on counts)
lda = LatentDirichletAllocation(
    n_components=k,
    learning_method="batch",  # or "online" for large data
    max_iter=20,
    random_state=42,
    evaluate_every=5          # enables perplexity during fit
)
lda_doc_topic = lda.fit_transform(X_count)   # doc-topic distributions
lda_topic_word = lda.components_             # topic-word counts

# B) NMF (on TF-IDF)
nmf = NMF(
    n_components=k,
    init="nndsvda",
    solver="cd",
    beta_loss="frobenius",   # try 'kullback-leibler' as well
    l1_ratio=0.0,
    max_iter=400,
    random_state=42
)
nmf_doc_topic = nmf.fit_transform(X_tfidf)   # doc-topic strengths
nmf_topic_word = nmf.components_


In [ ]:
df["topic_lda"] = lda_doc_topic.argmax(axis=1)
df["topic_nmf"] = nmf_doc_topic.argmax(axis=1)

# Optional: normalize to get proportions (0–1)
lda_doc_prop = normalize(lda_doc_topic, norm="l1", axis=1)
nmf_doc_prop = normalize(nmf_doc_topic, norm="l1", axis=1)


In [ ]:
# Example: topic volume
lda_counts = df["topic_lda"].value_counts().sort_index()
nmf_counts = df["topic_nmf"].value_counts().sort_index()
print("LDA topic counts:\n", lda_counts)
print(df.groupby("topic_lda")[["Likes","Retweets"]].mean().round(2))

- In LDA model, most topics have similar likes (40-45) and retweets (21-23), except `bitterness` gets lower likes (34) and retweets (17). This may suggest that the users engage less with negative topics.

In [ ]:
print("NMF topic counts:\n", nmf_counts)
print(df.groupby("topic_nmf")[["Likes","Retweets"]].mean().round(2))

- In NMF model, most topics have similar likes (40-45) and retweets (21-23), except anxiety gets lower likes (33) and retweets (17). This may suggest that the users engage less with negative topics.

In [ ]:
def topic_labels(H, vocab, topn=4):
    labels = []
    for comp in H:
        top_idx = comp.argsort()[::-1][:topn]
        labels.append(" / ".join(vocab[top_idx]))
    return labels

lda_labels = ['new', 'friend', 'nature', 'world', 'art', 'joy', 'community', 'symphony', 'bitterness', 'dreams']
nmf_labels = ['life', 'joy', 'serenity', 'gratitude', 'friends', 'nature', 'local', 'project', 'anxiety', 'symphony']

# Attach pretty labels for NMF
df["topic_nmf_label"] = df["topic_nmf"].map(dict(enumerate(nmf_labels)))
df["topic_lda_label"] = df["topic_lda"].map(dict(enumerate(lda_labels)))

In [ ]:
import plotly.express as px

# Topic by sentiment (stacked)
fig = px.histogram(df, x="topic_nmf_label", color="mapped_sentiment",
                       barmode="stack", title="Topic × Sentiment (NMF)")
fig.update_xaxes(categoryorder="total descending")
fig.show()


- In NMF, the topics that include mostly positive sentiment include nature, friends, gratitude, project, local and joy. Symphony, life, and anxiety is more balanced, with anxiety having one more negative sentiment than positive. 

In [ ]:
fig = px.histogram(df, x="topic_lda_label", color="mapped_sentiment",
                       barmode="stack", title="Topic × Sentiment (LDA)")
fig.update_xaxes(categoryorder="total descending")
fig.show()

- In LDA, the topics that include mostly positive sentiment include new, art, dreams, nature, joy, community and friend. Symphony, bitterness and world have more negative sentiment than positive.

In [ ]:
df.to_parquet('df_pq.parquet')

## Step 6. User Engagment Analaysis
Section 2 and 5 showed some exploratory results,
- Time of day → higher engagement in night/evening.
- Country → some stand out with higher likes/retweets.
- Sentiment/Topics → clear positive > negative difference.
- Hashtags → certain ones spike engagement.
  
In this section, we try to answer the following question.
- Given a post’s text (sentiment, topic, hashtags, time, country), can we predict its engagement (likes, retweets)?

We perform a classification task, dividing the likes into low, medium, and high. We first train a baseline random forest model, then proceed with gradient boosting.

### Baseline Random Forest
#### Preprocessing

In [ ]:
def bin_likes(x):
    if x <= 20:
        return "low"
    elif x <= 60:
        return "medium"
    else:
        return "high"

df["likes_bin"] = df["Likes"].apply(bin_likes)
print(df["likes_bin"].value_counts())


#### Select Features

In [ ]:
# Simplify time: group hours into bins
def hour_to_bin(h):
    if 5 <= h < 12:
        return "morning"
    elif 12 <= h < 17:
        return "afternoon"
    elif 17 <= h < 21:
        return "evening"
    else:
        return "night"

df["hour_bin"] = df["Hour"].apply(hour_to_bin)

# Hashtags: explode and keep top 20
tags_long = df.explode("hashtags_original")
tags_long["hashtags_original"] = tags_long["hashtags_original"].str.lower().str.strip().str.lstrip("#")
top_tags = tags_long["hashtags_original"].value_counts().head(20).index

for t in top_tags:
    df[f"tag_{t}"] = df["hashtags_original"].apply(lambda tags: int(t in [str(x).lower().lstrip("#") for x in (tags if isinstance(tags,list) else [])]))


#### Split Data

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["mapped_sentiment","topic_nmf","Country","hour_bin"] + [f"tag_{t}" for t in top_tags]]
y = df["likes_bin"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


#### Build Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

categorical = ["mapped_sentiment","topic_nmf","Country","hour_bin"]
hashtag_cols = [f"tag_{t}" for t in top_tags]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("tags", "passthrough", hashtag_cols)
    ]
)

clf = Pipeline(steps=[
    ("pre", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=42, class_weight="balanced"
    ))
])


#### Train and Evaluate

In [ ]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


- The accuracy is 0.8, suggesting the model predicts 80% of all test data correctly.
- The precision, recall and f1-score are low for high and low like categories, and high for the medium category. One explanation is a lack of data in the high and low categories. The model “plays it safe” by predicting the majority (medium) class, which boosts accuracy but lowers performance on rare categories.

#### Inspect Feature Importance

In [ ]:
# Get feature names after preprocessing
ohe = clf.named_steps["pre"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(categorical)
all_names = np.concatenate([cat_names, hashtag_cols])

importances = clf.named_steps["model"].feature_importances_
idx = np.argsort(importances)[::-1][:20]  # top 20

plt.figure(figsize=(10,6))
plt.barh(np.array(all_names)[idx][::-1], importances[idx][::-1])
plt.title("Top Feature Importances (Random Forest)")
plt.show()


- In this model, the most important features are positive and negative sentiment, topic, and whether the post was in the afternoon, which may be a strong indicator for medium number of likes. Some country indicators such as Canada, USA, UK are also important to generate the predictions.

Then we conduct a linear regression and then XGBoost regression to predict log likes.
### Regression on Log Likes

In [ ]:
df["likes_log"] = np.log1p(df["Likes"])

X = df[["mapped_sentiment","topic_nmf","Country","hour_bin"] + [f"tag_{t}" for t in top_tags]]
y = df["likes_log"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=df["likes_bin"], test_size=0.2, random_state=42
)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

linreg = Pipeline(steps=[
    ("pre", preprocessor),
    ("model", Ridge(alpha=1.0))  # L2 regularization
])

linreg.fit(X_train, y_train)
y_pred = linreg.predict(X_test)

print("Linear Regression Results")
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))
print("R^2:", r2_score(y_test, y_pred))


In [ ]:
import xgboost as xgb

# Build pipeline
xgb_reg = Pipeline(steps=[
    ("pre", preprocessor),
    ("model", xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ))
])

xgb_reg.fit(X_train, y_train)
y_pred_xgb = xgb_reg.predict(X_test)

print("XGBoost Regression Results")
print("MSE:", mean_squared_error(y_test, y_pred_xgb))
print("RMSE:", mean_squared_error(y_test, y_pred_xgb))
print("R^2:", r2_score(y_test, y_pred_xgb))


- Linear regression has lower MSE and RMSE. It captures the main trend with fewer parameters, performs robustly on this dataset size.
- XGBoost regression has higher R^2, picking up more non-linear interactions (e.g., sentiment × country, time × topic).

In [ ]:
# Get feature names after preprocessor
ohe = xgb_reg.named_steps["pre"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(categorical)
all_names = np.concatenate([cat_names, hashtag_cols])

import matplotlib.pyplot as plt
import numpy as np

booster = xgb_reg.named_steps["model"].get_booster()
importance = booster.get_score(importance_type="weight")

# Align with features
feat_imp = {all_names[int(k[1:])]: v for k,v in importance.items()}  # features like f0,f1,...

sorted_imp = sorted(feat_imp.items(), key=lambda x: x[1], reverse=True)[:20]
names, scores = zip(*sorted_imp)

plt.figure(figsize=(10,6))
plt.barh(names[::-1], scores[::-1])
plt.title("XGBoost Feature Importance (Top 20)")
plt.show()


- Some important features include sentiment, time bin, and countries like Canada, US, and UK.
- These align with the descriptive EDA: positive sentiment, evening posts, and certain geographies drive engagement.

Overall, Classification (RF) is useful for a coarse-grained view (low/medium/high), but struggles with minority classes due to data imbalance.
Regression gives finer-grained predictions of engagement magnitude. Linear regression excels in error minimization. XGBoost provides richer variance explanation and highlights non-linear effects. The features are consistent across all models (sentiment, time, country) reinforce their importance.

## Step 7. Conclusions

In this project, we explored a Kaggle dataset of social media posts with rich metadata including text, sentiment labels, hashtags, time and geography information, as well as engagement indicators such as likes and retweets.

Through exploratory data analysis, we examined posting patterns across time zones and countries, the distribution of sentiments, and the role of hashtags. This revealed broad engagement trends, such as higher interaction with positive content and evening/nighttime posting.

We applied sentiment analysis using both traditional machine learning (logistic regression with TF–IDF) and deep learning (LSTM). Logistic regression consistently outperformed LSTM on this dataset (~700 samples), likely due to the effectiveness of sparse n-gram features in low-data settings. Nevertheless, the LSTM complemented the logistic model by retrieving more neutral/other categories.

Using topic modeling (LDA and NMF), we uncovered latent themes such as community, friendship, nature, art, joy, and anxiety. While some topics were stable across models (e.g., “nature” and “community”), others reflected different partitions of the corpus. Linking topics to sentiment and engagement, we observed that positive themes generally attracted higher interaction, while negative themes (e.g., bitterness, anxiety) underperformed.

Finally, in user engagement analysis, we studied how sentiment, topic, time, hashtags, and country influenced likes and retweets. Groupby analysis showed that positive sentiment, evening posting times, and certain hashtags correlated with higher engagement. Predictive modeling (classification and regression) further confirmed that sentiment, posting time, and country were the most important features. Logistic regression minimized error well, while XGBoost captured non-linear interactions and explained more variance.

Overall, this study demonstrates how combining exploratory analysis, supervised learning, and unsupervised learning can provide a comprehensive understanding of social media dynamics. Positive, community-oriented content consistently drives engagement, while posting time and geography amplify visibility. Methodologically, classical models remain strong baselines in low-data settings, while topic modeling and regression add valuable interpretability.

Future work can include using more modern methods like BERTweet and predicting multiple outputs such as likes and retweets together.